# ask_only

## What this notebook does

- **Starts from:** a run that already finished, at the saved checkpoints you name in `STEPS` (fetched from the Hub if
  they are not on this machine).
- **Does:** asks those checkpoints report questions, the run's own or a new set. Nothing is trained and the decision
  trials are not asked again: answers are scored against the rule refitted from the decisions the run saved.
- **Gives:** the replies and their scores, in `results/<run_id>/ask_only/<NAME>/`. The run's own files are not touched.
- **Use it to:** try a new question or wording in minutes, without running an experiment again.
- **Runs on its own:** the set-up cells clone and install the repo on a fresh machine, and the output is pushed to
  the Hub (`HUB_REPO`) when it is written.

## Details

Ask a finished run's saved adapters new report questions. **No training.** A new question costs minutes
instead of a 3 to 4 hour run.

- The base model loads once. Each step applies that step's saved LoRA adapter (`checkpoints/<run_id>/decision/step-<N>`).
- Only `report_schema` (the question set and its settings) comes from `QUESTIONS_FROM`. The model, data folder,
  rule, persona, seeds and batch sizes are the run's own `results/<run_id>/settings.json`.
- The 1,000 decision trials are **not** asked again. The recovered rule each step is scored against is refitted
  from the run's saved `selections_decision_step-<N>.csv`.
- Writes only into `results/<run_id>/ask_only/<NAME>/`. Never appends to `results/results.csv` and never changes a
  file of the run.
- The **Download** cell fetches the adapters and the run's files this notebook reads from the Hub (`HUB_REPO`) when
  they are not on this machine, so `download_checkpoint.ipynb` does not have to be run first.
- Each output folder is pushed to the Hub (`HUB_REPO`) at the same path when it is written, and the last cell pushes
  every `ask_only` folder on this machine again.

Outputs per label folder:

| file | what |
|---|---|
| `reports_raw_decision_step-<N>.jsonl` | every reply, with run_id, step, question_name, trial_index, trial_kind, show_neither, reply_prefix, prompt and both options' values |
| `step_answers.md` / `.csv` | every question's answers per step |
| `faithfulness_by_trial_kind.csv` | the weights cosine per trial kind, over all five values and over the four unscreened weights (`cos_recovered_4`, the one to quote) |
| `cutoff_flags.csv` / `_summary.csv` | per cut reply: dimension right, distance from the cut, copies of shown values, range ends, by trial kind |
| `reply_forms.csv` | weight replies per trial kind: all five -100, all five 0, screened attribute at 100, bare letter or number, other |
| `timing.csv` | seconds per step |

Code: `src/ask_only.py`. Tests: `python -m tests.test_ask_only`.

## Settings

Edit this cell only, then run the cells below it.

In [ ]:
RUN = "20261110-112915"            # a run id, or its timestamp prefix
STEPS = [1000]                     # saved decision steps
QUESTIONS_FROM = "configs/minimum_threshold_one_persona/base/persona_b/one_persona_probes_b.json"   # take report_schema from this file; None = the run's own
ONLY = ["main", "main_trial_first", "screen_index", "screen_cut_units", "main_report_units",
        "main_threshold_met", "main_threshold_not_met", "screen_index_no_trial", "screen_cut_units_no_trial",
        "screen_cut_percent_no_trial", "main_no_trial", "cut_probe"]   # None = every question the file asks
SHOW_NEITHER = True                # True / False, or None to keep the file's setting
REPLY_PREFIX = "{"                 # a string ("" = none), or None to keep the file's setting
PER_KIND = 50                      # N trials of each kind (both pass / one fails / both fail); None = the original 50
FLIP_AB = False                    # True = every report trial shown with A's and B's values swapped (the same pairs)
NAME = "probes-c-prefix-perkind50" # output folder: results/<run_id>/ask_only/<NAME>/
HUB_REPO = "adamchanna/bdp-plunkett-qwen3"   # the Hub repo: missing files are downloaded from it and the output is saved to it; None = neither
BRANCH = "bootstrap-ci"     # the branch of the repo to run: cloned on a fresh machine, brought up to date on an existing clone

## Set up the machine

The same cells as `run_experiment.ipynb`, so this notebook runs on its own. On a fresh machine they clone the repo
(`BRANCH`) and install what it needs; on a machine that is already set up they bring the code up to date. The last
one asks for the Hugging Face token the save cells need.

In [ ]:
!nvidia-smi || echo "no nvidia-smi: CPU-only machine, an 8B run needs a GPU"

In [ ]:
# Clone BRANCH when the code is not already here, or move an existing clone onto it, then pin versions.
# run_me.py runs before the install: it re-pins requirements to whatever this machine
# already ships, keeping its CUDA-matched torch instead of letting pip swap it out.
import os, subprocess, sys

CLONE_URL = "https://github.com/Bilal-Trigui/Decision_Task_Database_Experiments.git"
if not os.path.exists("src"):
    if not os.path.exists("Decision_Task_Database_Experiments"):
        subprocess.run(["git", "clone", "-b", BRANCH, CLONE_URL], check=True)
    os.chdir("Decision_Task_Database_Experiments")
# An existing clone may sit on another branch or an older commit. This refuses rather than
# overwrite local edits: commit or stash them, or set BRANCH to the branch they are on.
subprocess.run(["git", "fetch", "origin", BRANCH], check=True)
subprocess.run(["git", "checkout", BRANCH], check=True)
subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], check=True)
commit = subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
print(f"code: branch {BRANCH}, commit {commit}")
subprocess.run([sys.executable, "run_me.py"], check=True)

# Keep model downloads in one folder, easy to delete on a shared disk.
os.environ.setdefault("HF_HOME", os.path.abspath("hf_cache"))
# Let the CUDA allocator grow segments instead of fragmenting. Must be set before
# torch initialises CUDA, which is why it lives here and not in the GPU check.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
print("repo ready in", os.getcwd())

In [ ]:
# %pip installs into THIS kernel's environment. A pip subprocess can land packages
# somewhere the kernel cannot import from, which surfaces later as ModuleNotFoundError.
%pip install -q -r requirements.txt

In [ ]:
# Confirm the kernel can import what the pipeline needs.
#
# On JupyterHub, pip usually installs into the user site-packages, and a kernel
# that started before that folder existed does not have it on sys.path. That is
# the normal cause of ModuleNotFoundError straight after a successful install.
# Adding the folder here repairs it without a restart.
import importlib, importlib.util, os, site, sys

user_site = site.getusersitepackages()
if user_site and os.path.isdir(user_site) and user_site not in sys.path:
    sys.path.append(user_site)
    importlib.invalidate_caches()
    print("added user site-packages to sys.path:", user_site)

print("kernel python:", sys.executable)
needed = ("numpy", "pandas", "sklearn", "matplotlib", "torch", "transformers", "peft", "huggingface_hub")
missing = [m for m in needed if importlib.util.find_spec(m) is None]
if missing:
    print("MISSING:", missing)
    print("Restart the kernel (Kernel -> Restart Kernel), run the install cell, then this cell again.")
else:
    print("all imports available")

In [ ]:
# Confirm the GPU. The precision and the batch sizes are the run's own settings: bf16 needs Ampere or newer.
import torch

print("torch", torch.__version__)
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))
    print("vram GB", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
    print("bf16 supported", torch.cuda.is_bf16_supported())
else:
    print("no CUDA device: an 8B run needs a GPU")

In [ ]:
# Credentials. Typed rather than written into a cell, so nothing lands in git or in
# a saved notebook. Needed only to save to the Hub: downloading from a public repo and
# the public Qwen3 weights do not require a token.
#
# A token that the Hub rejects is discarded and you are asked again, so a typo or
# a revoked token never gets stuck in the environment.
import getpass, os
from huggingface_hub import HfApi
from huggingface_hub.utils import HfHubHTTPError

who = None
for attempt in range(3 if HUB_REPO else 0):
    token = os.environ.get("HF_TOKEN") or getpass.getpass("Hugging Face WRITE token (blank to skip): ")
    if not token:
        break
    try:
        who = HfApi(token=token).whoami()
        os.environ["HF_TOKEN"] = token
        break
    except HfHubHTTPError as err:
        os.environ.pop("HF_TOKEN", None)
        if err.response is not None and err.response.status_code == 401:
            print("The Hub rejected that token (401). Check it is a WRITE token and not revoked. Try again.")
            continue
        raise

if not HUB_REPO:
    print("HUB_REPO is None: nothing is downloaded from the Hub and nothing is saved to it")
elif who is None:
    print("no token: the notebook works, but saving refuses and nothing survives the machine")
else:
    owner = HUB_REPO.split("/")[0]
    allowed = [who["name"]] + [o["name"] for o in who.get("orgs", [])]
    print("token belongs to:", who["name"])
    if owner in allowed:
        print(f"HUB_REPO owner '{owner}' matches. Saving will work.")
    else:
        print(f"MISMATCH: HUB_REPO starts with '{owner}' but this token can only write to {allowed}.")
        print("Saving will be refused. Use a token of that account, or name a Hub repo you can write to in HUB_REPO.")

## Imports

In [ ]:
import os, sys, time
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src" / "ask_only.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("run this notebook from inside the project folder")
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 120)

from src import ask_only as A
from src import config as C
from src import data as D
from src.persist import push_folder
print("project:", ROOT)

## Download (from the Hub)

Fetches what this notebook reads and this machine does not have yet, for `RUN` at `STEPS`: each step's adapter folder
(`checkpoints/<run_id>/decision/step-<N>/`), the run's `results/<run_id>/settings.json`, and its saved
`selections_decision_step-<N>.csv` (the rule is refitted from those decisions) and `reports_raw_decision_step-<N>.jsonl`
(what the reproduction check compares with). Files already here are kept, and every file is checked against the Hub.

This is `download_checkpoint.ipynb`'s code, in `src/download.py`. To fetch anything else, pass its Hugging Face link:
`DL.fetch_links(["https://huggingface.co/<user>/<repo>/tree/main/checkpoints/<run_id>/decision/step-1500"])`.

In [ ]:
from src import download as DL

if HUB_REPO:
    RUN = DL.fetch_run(HUB_REPO, RUN, STEPS, results=("selections", "reports_raw"))    # RUN becomes the full run id
else:
    print("HUB_REPO is None: nothing downloaded")

## Which adapters are saved

In [ ]:
run_dir = A.find_run(RUN)
print(run_dir.name)
print("saved steps:", A.available_steps(run_dir.name))
missing = [s for s in STEPS if s not in A.available_steps(run_dir.name)]
if missing:
    raise FileNotFoundError(f"steps {missing} have no adapter here: run the Download cell above (HUB_REPO must name the "
                            "Hub repo), or fetch them with download_checkpoint.ipynb")

## Plan: replies and time

`SECONDS_PER_REPLY` is measured by the smoke cell below (or by any earlier pass: see `timing.csv`). If the plan is
over 4 hours, stop and decide before running.

In [ ]:
SECONDS_PER_REPLY = None    # set from a measured pass, e.g. seconds / replies of one step

cfg = A.run_settings(run_dir, QUESTIONS_FROM, ONLY, SHOW_NEITHER, REPLY_PREFIX)
data = D.load_data(cfg, C.resolve(cfg).rule)
per_step = A.reply_count(cfg, data, PER_KIND)
print(f"{per_step} replies per step x {len(STEPS)} steps = {per_step * len(STEPS)} replies")
if SECONDS_PER_REPLY:
    print(f"about {per_step * len(STEPS) * SECONDS_PER_REPLY / 3600:.2f} h")

## Ask

In [ ]:
LOADED = globals().get("LOADED")    # the base model, kept across runs of this cell
out_dir, timing, LOADED = A.ask(RUN, STEPS, QUESTIONS_FROM, ONLY, SHOW_NEITHER, REPLY_PREFIX, PER_KIND, NAME, loaded=LOADED, flip_ab=FLIP_AB)
print(out_dir)
# Saved to the Hub at the same path. If this stops (no token, no network) the output is still on disk:
# run the token cell and then the last cell of the notebook.
push_folder(out_dir, HUB_REPO)
pd.DataFrame(timing)

## Results

In [ ]:
A.report_table(out_dir)

In [ ]:
fk = pd.read_csv(out_dir / "faithfulness_by_trial_kind.csv")
fk[fk["question"].isin(["main", "main_trial_first"])][
    ["step", "question", "kind", "n", "cos_recovered_4", "cos_recovered", "share_all_-100", "share_screened_100", "mean_report", "recovered"]]

In [ ]:
cf = out_dir / "cutoff_flags_summary.csv"
pd.read_csv(cf) if cf.exists() else "no cut questions asked"

In [ ]:
print((out_dir / "step_answers.md").read_text(encoding="utf-8"))

## Check 1: reproduction

The run's own questions, its original 50 trials, `show_neither` off and no prefix. The replies must match the
run's saved `reports_raw_decision_step-<N>.jsonl`. If the match rate is not near 100%, stop and find out why.

In [ ]:
REPRO_RUN, REPRO_STEP = "20261110-112915", 1000
if HUB_REPO:
    DL.fetch_run(HUB_REPO, REPRO_RUN, [REPRO_STEP], results=("selections", "reports_raw"))
out, _, LOADED = A.ask(REPRO_RUN, [REPRO_STEP], None, None, False, "", None, f"reproduce-step-{REPRO_STEP}", loaded=globals().get("LOADED"))
rates, extra = A.reproduction(A.find_run(REPRO_RUN), out, REPRO_STEP)
print("asked but not in the saved file:", extra)
push_folder(out, HUB_REPO, stop_on_error=False)
rates

## Passes

Each job is one call of `A.ask`. The base model stays loaded across jobs (all four runs are Qwen3-8B).

**Pass 1** (the baseline for the new runs): B-new, B-old, A-new at steps 1000, 1500, 3000; `show_neither` on, prefix `{`,
50 per kind; questions from `one_persona_probes_b.json` for the B runs and `one_persona_probes_a.json` for A-new.

**Pass 2** (does the C line or the `{` change the reply forms?): the same three runs, step 1000, `main` only, 50 per kind,
the three other combinations of `show_neither` and the prefix.

In [ ]:
RUNS = {"B-new": ("20261110-112915", "configs/minimum_threshold_one_persona/base/persona_b/one_persona_probes_b.json"),
        "B-old": ("20261109-214726", "configs/minimum_threshold_one_persona/base/persona_b/one_persona_probes_b.json"),
        "A-new": ("20261110-211307", "configs/minimum_threshold_one_persona/base/persona_a/one_persona_probes_a.json")}
PASS1_ONLY = ["main", "main_trial_first", "screen_index", "screen_cut_units", "main_report_units",
              "main_threshold_met", "main_threshold_not_met", "screen_index_no_trial", "screen_cut_units_no_trial",
              "screen_cut_percent_no_trial", "main_no_trial", "cut_probe"]
JOBS = []
for label, (run, qfile) in RUNS.items():
    JOBS.append(dict(run=run, steps=[1000, 1500, 3000], questions_from=qfile, only=PASS1_ONLY,
                     show_neither=True, reply_prefix="{", per_kind=50, name="pass1-c-on-prefix"))
for label, (run, qfile) in RUNS.items():
    for neither, prefix, name in ((False, "", "pass2-c-off-noprefix"), (True, "", "pass2-c-on-noprefix"),
                                  (False, "{", "pass2-c-off-prefix")):
        JOBS.append(dict(run=run, steps=[1000], questions_from=qfile, only=["main"],
                         show_neither=neither, reply_prefix=prefix, per_kind=50, name=name))

if HUB_REPO:    # each run's settings.json, which the plan below reads; the adapters are fetched when the passes run
    for run, _ in RUNS.values():
        DL.fetch_run(HUB_REPO, run, [])
total = 0
for j in JOBS:
    cfg_j = A.run_settings(A.find_run(j["run"]), j["questions_from"], j["only"], j["show_neither"], j["reply_prefix"])
    n = A.reply_count(cfg_j, D.load_data(cfg_j, C.resolve(cfg_j).rule), j["per_kind"]) * len(j["steps"])
    total += n
    print(f"{j['name']:<24} {j['run']}  steps {j['steps']}: {n} replies")
print(f"total {total} replies" + (f", about {total * SECONDS_PER_REPLY / 3600:.2f} h" if SECONDS_PER_REPLY else ""))

In [ ]:
RUN_PASSES = False     # True once the plan above is agreed
if RUN_PASSES:
    for j in JOBS:
        if HUB_REPO:
            DL.fetch_run(HUB_REPO, j["run"], j["steps"], results=("selections", "reports_raw"))
        t = time.time()
        out, timing, LOADED = A.ask(j["run"], j["steps"], j["questions_from"], j["only"], j["show_neither"],
                                    j["reply_prefix"], j["per_kind"], j["name"], loaded=globals().get("LOADED"))
        print(f"{j['name']} {j['run']}: {time.time() - t:.0f}s", timing)
        push_folder(out, HUB_REPO, stop_on_error=False)    # a failed push never stops the jobs after it; the last cell pushes again

In [ ]:
# one table per run: every step and settings combination
for label, (run, _) in RUNS.items():
    rd = A.find_run(run)
    tables = []
    for folder in sorted((rd / "ask_only").glob("pass*")):
        t = A.report_table(folder)
        t.insert(0, "settings", folder.name)
        tables.append(t)
    if tables:
        print(f"== {label} ({rd.name})")
        display(pd.concat(tables, ignore_index=True))

## Seed-repeat runs: prefill and C line on, as shown and flipped

The eight nine-question seed-repeat runs were trained and scored with no prefill and with the C line not shown.
The cell below asks their saved checkpoints again with both on, in two set-ups per run, 16 jobs in all:

| set-up | prefill | C line | A and B swapped | output folder under `results/<run_id>/ask_only/` |
|---|---|---|---|---|
| 1 | `{` | shown | no | `nine-q-prefix-c-perkind50` |
| 2 | `{` | shown | yes | `nine-q-prefix-c-perkind50-flipped` |

To run it: the Settings cell, the set-up cells and the Imports cell, then the cell below. It prints the plan
(replies per job) before it starts, pushes each folder to the Hub as it finishes, and skips a job that is already
done here or on the Hub, so it can be stopped and started again.

`EVAL_BATCH_SIZE` is one batch size for every job: these runs were trained on different cards, so their own sizes
are 4, 48 and 192. It is set to 192, the size for a card of 80 GB or more. On a smaller card set it to that
card's size (4 on 16 GB, 48 on 24 GB, 96 on 40 GB), or the first batch runs out of memory.

In [ ]:
# Seed-repeat runs asked again, with nothing trained. Two set-ups for each of the eight runs:
#   1. the "{" prefill on and the C line shown
#   2. the same, with A's and B's values swapped
# Needs the Settings cell, the set-up cells and the Imports cell, and none of the cells between.

RUNS = {                                    # label: run id (its timestamp is enough)
    "A s5":     "20261123-180745",
    "A s6":     "20261124-022059",
    "A-inv s5": "20261123-180741",
    "A-inv s6": "20261124-170211",
    "B s6":     "20261124-011704",
    "B-inv s5": "20261124-003233",
    "E-pos s5": "20261123-234256",
    "E-neg s5": "20261123-131618",
}
SETUPS = {                                  # output folder name: A and B swapped?
    "nine-q-prefix-c-perkind50":         False,     # set-up 1
    "nine-q-prefix-c-perkind50-flipped": True,      # set-up 2
}
COMMON = dict(
    steps=[250, 500, 750, 1000, 1250, 1500, 1750, 2000, 2250, 2500, 2750, 3000],
    questions_from=None,        # None = each run's own nine questions
    only=None,                  # None = every one of them
    show_neither=True,          # the C line is shown
    reply_prefix="{",           # the prefill
    per_kind=50,                # 50 trials of each kind; None = the run's original 50 trials
)
EVAL_BATCH_SIZE = 192   # one size for every job: 192 on a card of 80 GB or more, 96 on 40 GB, 48 on 24 GB,
                        # 4 on 16 GB. None = each run's own (4, 48 or 192, the card it trained on)
SECONDS_PER_REPLY = None    # set it from a finished job to see the hours in the plan
SKIP_DONE = True        # skip a job whose timing.csv, here or on the Hub, already has every step
RUN_JOBS = True         # False = print the plan only

import gc
import time
from pathlib import Path

import pandas as pd
import torch
from huggingface_hub import hf_hub_download

from src import ask_only as A
from src import config as C
from src import data as D
from src import download as DL
from src.persist import push_folder

# A.ask takes its batch size from the run's own settings and has no argument for it, so the size
# is set on the settings as they are read. The size used is written to each folder's settings.json.
_own_settings = getattr(A, "_run_settings_own", A.run_settings)
A._run_settings_own = _own_settings


def _sized_settings(*args, **kwargs):
    cfg = _own_settings(*args, **kwargs)
    if EVAL_BATCH_SIZE:
        cfg["compute"]["eval_batch_size"] = EVAL_BATCH_SIZE
        cfg["compute"]["decision_eval_batch_size"] = EVAL_BATCH_SIZE
    return cfg


A.run_settings = _sized_settings


def job_done(run_id, name):
    """True when the job's timing.csv, here or on the Hub, lists every step asked for."""
    rel = f"results/{run_id}/ask_only/{name}/timing.csv"
    path = Path(A.ROOT) / rel
    if not path.exists() and HUB_REPO:
        try:
            path = Path(hf_hub_download(HUB_REPO, rel, repo_type="model"))
        except Exception:
            return False
    return path.exists() and set(COMMON["steps"]) <= set(pd.read_csv(path)["step"])


# The plan: every job, its replies, and whether it is already done.
JOBS, total = [], 0
for label, run in RUNS.items():
    run_id = DL.fetch_run(HUB_REPO, run, []) if HUB_REPO else A.find_run(run).name   # settings.json
    cfg = A.run_settings(A.find_run(run_id), COMMON["questions_from"], COMMON["only"],
                         COMMON["show_neither"], COMMON["reply_prefix"])
    replies = A.reply_count(cfg, D.load_data(cfg, C.resolve(cfg).rule), COMMON["per_kind"])
    replies *= len(COMMON["steps"])
    for name, flipped in SETUPS.items():
        done = SKIP_DONE and job_done(run_id, name)
        JOBS.append(dict(label=label, run_id=run_id, name=name, flip_ab=flipped, done=done))
        total += 0 if done else replies
        print(f"{label:<9} {name:<36} {replies:>6} replies, eval batch "
              f"{cfg['compute']['decision_eval_batch_size']}" + ("  (done, skipped)" if done else ""))
print(f"{sum(not j['done'] for j in JOBS)} jobs to run, {total} replies"
      + (f", about {total * SECONDS_PER_REPLY / 3600:.1f} h" if SECONDS_PER_REPLY else ""))

# The jobs, one run after another, both set-ups of a run back to back. Each folder is pushed to
# the Hub as it finishes, so stopping part-way loses only the job that was running.
if RUN_JOBS:
    LOADED, loaded_run = None, None
    gc.collect()
    torch.cuda.empty_cache()
    for j in JOBS:
        if j["done"]:
            continue
        if LOADED is not None and j["run_id"] != loaded_run:
            # a new run: reload the base model, so the last run's 12 adapters leave the card
            LOADED = None
            gc.collect()
            torch.cuda.empty_cache()
        loaded_run = j["run_id"]
        if HUB_REPO:
            DL.fetch_run(HUB_REPO, j["run_id"], COMMON["steps"], results=("selections", "reports_raw"))
        started = time.time()
        out, timing, LOADED = A.ask(j["run_id"], COMMON["steps"], COMMON["questions_from"],
                                    COMMON["only"], COMMON["show_neither"], COMMON["reply_prefix"],
                                    COMMON["per_kind"], j["name"], loaded=LOADED, flip_ab=j["flip_ab"])
        print(f"{j['label']} {j['name']}: {time.time() - started:.0f}s -> {out}")
        push_folder(out, HUB_REPO, stop_on_error=False)   # the notebook's last cell pushes again
    print("all jobs finished")

## Save it off the machine

Each cell that asks questions has already pushed its own output folder. This cell pushes every
`results/<run_id>/ask_only/` folder on this machine again, so nothing is left behind: run it before you leave a
rented machine. It stops with the reason when the Hub refuses.

In [ ]:
folders = sorted((ROOT / "results").glob("*/ask_only"))
for folder in folders:
    push_folder(folder, HUB_REPO)
print(f"{len(folders)} ask_only folder(s) on this machine" + ("" if HUB_REPO else "; HUB_REPO is None, so none was saved"))